# Project K, equal battery: Qwen2.5-Omni-3B runs the completed Series One battery

**Why this notebook exists.** Kobby's equal-tests ruling (Nov 2,
2026): every model goes through the same Series One battery before
any post speaks across models. This model still owes the proper swap cells, the bed, the blackout, and the muted pairs. This notebook runs the
whole battery for Qwen2.5-Omni-3B, identical to the other three
equal-battery notebooks: same files, same questions byte for byte,
same repeats, same 33-column results schema as the Series Two runs.

**The battery (41 asks).**

- **Part A (20 asks).** The repaired JSON question on the three
  real clips (vacuum, jar, eggs) and all seven swap files, two
  repeats each.
- **Part A2 (12 asks).** The same question on the three real
  clips in two built versions: muted (picture, no sound) and
  blackout (sound, no picture), two repeats each.
- **Part B (6 asks).** The plain sounds question on the three
  matched bed files (the sentence alone, the sentence over beeps,
  the sentence over the vacuum bed), two repeats each.
- **Part B door (3 asks).** A transcription question on the same
  three matched files, one repeat each. The bed is judged per
  prompt: a model's bed rows are read only through prompts whose
  baseline (the sentence alone) returned the sentence. The door
  gives every model that baseline check in its strongest form.

**What is reused, unchanged.** Step 1 and Step 3 are this model's
Series Two ones, untouched: the Series One candidate list under the session Python: `Qwen/Qwen2.5-Omni-3B` in float16, then `Qwen/Qwen2.5-Omni-7B-AWQ`, the talker turned off on each, and the processor from the candidate's repo. Step 1 installs the stack into the session with transformers pinned at 4.52.4 and proves the pin. The runner's ask path,
feed handling, and feed proof are the Series Two ones. Only Step
2's file list and the battery are new.

**The feed proof rule.** Before every answer is taken, the runner
prints the proof for that exact feed, measured on the model's own
feed path. If a track that should carry sound decodes missing,
empty, short, or all zeros, the run hard-stops with a loud banner
and the rows so far are saved. Declared silence (the swap silence
file and the muted versions) is exempt by name in the job list and
is itself verified to be zeros. Nothing is padded quietly.

**How to run it.** Run the cells from the top, in order. Step 1
builds the environment and proves the stack. Step 2 copies the
thirteen files, builds the six muted and blackout versions,
measures them, and saves the built files into the Drive clips
folder. Step 3 writes the runner and runs its check mode (proof,
loading ladder, sanity). Step 4 runs the battery and saves
`equal_qwen_results.csv` in the Drive clips folder.

## Step 1 · Setup

The next cell installs the parts the model needs, with fixed version numbers
so the run is the same every time. It then prints the graphics chip it found.

**Before you run it:** the chip must be a T4. In the Colab menu, open
Runtime, then "Change runtime type", and pick the T4 GPU. On the free plan
this is the normal choice. If the cell prints "no GPU found", stop and change
the runtime type first. The model will not fit in plain memory.

The install takes a few minutes. That is normal.

One more setup step, new after run 2: the first code cell sets a memory
option (expandable segments) before torch loads. It lets the chip's memory
manager grow in pieces instead of reserving one big block, which helps when
the model already takes most of the chip. If you rerun cells in an old
session, restart the runtime once first, so the option takes effect from a
clean start.
**Series Two addition.** At the end, the cell proves the
stack it just installed: it prints the imported transformers
version and the file it loaded from, and stops the run if
the version is not the pinned 4.52.4. The other Series Two
notebooks prove their stacks the same way. This matters more
here than there, because the Qwen model runs under this
session's own Python: there is no private environment, so
these installed versions are the run's versions.


In [ ]:
# --- Setup -------------------------------------------------------------------
# First thing in this cell, before torch is loaded: ask the memory manager
# to use growing segments. This must be set BEFORE torch loads, which is
# why it sits above the torch import below. In a fresh runtime it takes
# effect. If you rerun this cell in a session where torch was already
# loaded, restart the runtime once and run from the top.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Fixed versions, so today's run matches tomorrow's run.
# soundfile and librosa are how the model's tools read the sound track
# out of each video file. Without them, the video would arrive silent,
# and this whole test would prove nothing.
import sys, subprocess

PACKAGES = [
    "transformers==4.52.4",
    "accelerate==1.8.1",
    "qwen-omni-utils==0.0.8",
    "soundfile==0.13.1",
    "librosa==0.11.0",
    "av",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PACKAGES,
               check=False)
print("install step done")

import torch
if torch.cuda.is_available():
    print("GPU found:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("no GPU found. Stop here. Set the runtime type to the T4 GPU,")
    print("then run this cell again.")
# --- Series Two addition: prove the installed stack ----------------------
# The model runs under this same session Python (the Qwen
# notebooks have no private environment), so the versions
# installed above are the versions the run uses. Prove them
# here, and stop if the transformers pin is not met.
import transformers
print("transformers version:", transformers.__version__)
print("transformers file:", transformers.__file__)
if transformers.__version__ != "4.52.4":
    raise SystemExit("STOP: transformers is "
                     + transformers.__version__
                     + ", not the pinned 4.52.4. Do not go on.")
print("stack proof passed: transformers 4.52.4, the Series One pin.")


## Step 2 · Get the clips; build the muted and blackout versions

Everything this run asks about lives in your Google Drive, in the
folder named `project-k-clips`. The next cell links your Drive,
finds that folder, and copies thirteen files to a plain work
folder here:

**The three real clips:**

1. `P02_07_5.mp4`: the vacuum clip.
2. `P32_01_54.mp4`: the jar clip.
3. `own-01.mp4`: your eggs recording.

**The seven swap files** (one still picture, seven soundtracks,
loudness matched at build time): `swap_silence.mp4`,
`swap_speech.mp4`, `swap_beeps.mp4`, `swap_water.mp4`,
`swap_vacuum.mp4`, `swap_sizzle.mp4`, `swap_mixed.mp4`.

**The three matched bed files:**

11. `matched_speech.mp4`: the spoken sentence alone, the baseline.
12. `matched_speech_beeps.mp4`: the sentence over a beep bed.
13. `matched_speech_vacuum.mp4`: the sentence over a vacuum bed.

If any of the thirteen is missing, the cell stops and names it.
Nothing is substituted for a missing file, the matched files
included: a bed test without its exact files is not the bed test.

**Standing rule:** a real clip longer than 20 seconds is cut to
its first 15 seconds (that is the eggs recording), the same
treatment as the earlier notebooks. The swap and matched files
are exact test files and are never cut.

**Then the cell builds six versions** from the real clips as they
stand in the work folder, and measures each one from its decoded
track before printing the measurement:

- **Muted** (`equal_muted_<name>`): the same picture, a silent
  track. The decoded track must measure as silence (RMS at or
  below -90 dB) at the source's duration, or the cell stops.
- **Blackout** (`equal_blackout_<name>`): a plain black picture
  at the clip's own size and length, the original sound
  untouched. The decoded track's duration and RMS must match the
  source clip's (within 0.5 seconds and 1.5 dB), or the cell
  stops.

Built files are saved back into the Drive clips folder, so every
model's run can ride the identical files.

Do not go on until the ready line names all of it: 3 real clips,
7 swap files, 3 matched files, 6 built versions.

In [ ]:
# --- Get the clips; build the muted and blackout versions -----------------------
# Copies thirteen files from the Drive clips folder, applies the
# standing cut rule to the real clips, then builds the six muted
# and blackout versions and measures every built file from its
# decoded track. A missing file stops the cell and names the
# file; a built file that decodes wrong stops the cell too.
import os, shutil, subprocess, json, math
import numpy as np

REAL = ["P02_07_5.mp4", "P32_01_54.mp4", "own-01.mp4"]
SWAP = ["swap_silence.mp4", "swap_speech.mp4", "swap_beeps.mp4",
        "swap_water.mp4", "swap_vacuum.mp4", "swap_sizzle.mp4",
        "swap_mixed.mp4"]
MATCHED = ["matched_speech.mp4", "matched_speech_beeps.mp4",
           "matched_speech_vacuum.mp4"]
WORK = "/content/clips"
DRIVE_CLIPS = "/content/drive/MyDrive/project-k-clips"
os.makedirs(WORK, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)

SOURCES = [DRIVE_CLIPS, "/content/project-k-clips"]

def bring(name):
    dest = os.path.join(WORK, name)
    if os.path.exists(dest):
        return True
    for folder in SOURCES:
        src = os.path.join(folder, name)
        if os.path.exists(src):
            shutil.copy(src, dest)
            print("copied", name, "from", folder)
            return True
    return False

missing = [name for name in REAL + SWAP + MATCHED if not bring(name)]
if missing:
    print("!" * 72)
    print("STOP: these files are missing from the Drive clips folder:")
    for name in missing:
        print("MISSING:", name)
    print("Nothing is substituted for a missing file. Add the exact")
    print("files to the project-k-clips folder, then run this cell again.")
    print("!" * 72)
    raise SystemExit("STOP: files missing in Step 2: "
                     + ", ".join(missing))

def duration_s(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", path],
        capture_output=True, text=True)
    try:
        return float(out.stdout.strip())
    except Exception:
        return None

def decode_mono(path, sr=48000):
    proc = subprocess.run(
        ["ffmpeg", "-v", "error", "-i", path, "-vn", "-ac", "1",
         "-ar", str(sr), "-f", "f32le", "-"],
        capture_output=True)
    if proc.returncode != 0:
        raise RuntimeError(f"decode failed for {path}: "
                           + proc.stderr.decode(errors="replace")[-500:])
    return np.frombuffer(proc.stdout, dtype=np.float32), sr

def rms_db(x):
    if len(x) == 0:
        return -120.0
    v = float(np.sqrt(np.mean(np.square(x, dtype=np.float64))))
    return 20.0 * math.log10(v) if v > 0 else -120.0

def save_both(local_path):
    if os.path.isdir(DRIVE_CLIPS):
        shutil.copy(local_path, os.path.join(
            DRIVE_CLIPS, os.path.basename(local_path)))
        print("saved to Drive clips folder:",
              os.path.basename(local_path))

def video_size(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=width,height",
         "-of", "csv=p=0", path],
        capture_output=True, text=True)
    try:
        w, h = out.stdout.strip().split(",")
        return int(w), int(h)
    except Exception:
        return None, None

# The standing cut rule, real clips only: longer than 20 seconds
# means cut to the first 15. Swap and matched files are exact
# test files and are never cut.
for name in REAL:
    path = os.path.join(WORK, name)
    d = duration_s(path)
    if d is not None and d > 20:
        short = os.path.join(WORK, "short_" + name)
        subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", path,
                        "-t", "15", "-c:v", "libx264",
                        "-pix_fmt", "yuv420p", "-c:a", "aac", short],
                       check=False)
        if os.path.exists(short):
            os.replace(short, path)
            print("cut", name, "to its first 15 seconds")

build_log = []

# ---- muted versions -------------------------------------------------------------
for name in REAL:
    src = os.path.join(WORK, name)
    out = os.path.join(WORK, "equal_muted_" + name)
    if not os.path.exists(out):
        proc = subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-i", src,
             "-f", "lavfi", "-i",
             "anullsrc=channel_layout=stereo:sample_rate=48000",
             "-map", "0:v:0", "-map", "1:a:0",
             "-c:v", "copy", "-c:a", "aac", "-ar", "48000",
             "-shortest", out])
        if proc.returncode != 0 or not os.path.exists(out):
            raise SystemExit(f"STOP: muted build failed for {name}")
    x, sr = decode_mono(out)
    entry = {"file": os.path.basename(out), "kind": "muted",
             "duration_s": round(len(x) / sr, 3),
             "audio_rms_db_decoded": round(rms_db(x), 2),
             "nonzero_share": round(float(np.count_nonzero(x) / len(x)), 4)}
    src_d = duration_s(src)
    if (src_d is None or abs(entry["duration_s"] - src_d) > 0.5
            or entry["audio_rms_db_decoded"] > -90.0):
        raise SystemExit(f"STOP: muted build for {name} decodes "
                         f"wrong: {entry}")
    build_log.append(entry)
    save_both(out)
    print("muted:", entry)

# ---- blackout versions ------------------------------------------------------------
for name in REAL:
    src = os.path.join(WORK, name)
    out = os.path.join(WORK, "equal_blackout_" + name)
    w, h = video_size(src)
    d = duration_s(src)
    if w is None or d is None:
        raise SystemExit(f"STOP: cannot read size or duration of {name}")
    if not os.path.exists(out):
        proc = subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-f", "lavfi",
             "-i", f"color=c=black:s={w}x{h}:r=25:d={d:.3f}",
             "-i", src, "-map", "0:v", "-map", "1:a",
             "-t", f"{d:.3f}", "-c:v", "libx264",
             "-pix_fmt", "yuv420p", "-c:a", "aac", "-ar", "48000",
             "-shortest", out])
        if proc.returncode != 0 or not os.path.exists(out):
            raise SystemExit(f"STOP: blackout build failed for {name}")
    xs, srs = decode_mono(src)
    x, sr = decode_mono(out)
    entry = {"file": os.path.basename(out), "kind": "blackout",
             "duration_s": round(len(x) / sr, 3),
             "audio_rms_db_decoded": round(rms_db(x), 2),
             "source_rms_db_decoded": round(rms_db(xs), 2),
             "nonzero_share": round(float(np.count_nonzero(x) / len(x)), 4)}
    if (abs(entry["duration_s"] - d) > 0.5
            or abs(entry["audio_rms_db_decoded"]
                   - entry["source_rms_db_decoded"]) > 1.5):
        raise SystemExit(f"STOP: blackout build for {name} decodes "
                         f"wrong: {entry}")
    build_log.append(entry)
    save_both(out)
    print("blackout:", entry)

if build_log:
    log_path = os.path.join(WORK, "equal_build_log.json")
    with open(log_path, "w") as fh:
        json.dump(build_log, fh, indent=2)
    if os.path.isdir(DRIVE_CLIPS):
        shutil.copy(log_path, os.path.join(DRIVE_CLIPS,
                                           "equal_build_log.json"))
    print("builds logged:", log_path)

print()
ready = True
for name in REAL + SWAP + MATCHED:
    path = os.path.join(WORK, name)
    if os.path.exists(path):
        print("ready:  ", name, f"({os.path.getsize(path) / 1e6:.1f} MB)")
    else:
        ready = False
        print("MISSING:", name)
for name in REAL:
    for kind in ("muted", "blackout"):
        path = os.path.join(WORK, f"equal_{kind}_" + name)
        if os.path.exists(path):
            print("ready:  ", f"equal_{kind}_" + name,
                  f"({os.path.getsize(path) / 1e6:.1f} MB)")
        else:
            ready = False
            print("MISSING:", f"equal_{kind}_" + name)
if not ready:
    print()
    print("!" * 72)
    print("STOP. Some files are missing. Check the Drive folder name is")
    print("exactly project-k-clips, then run this cell again.")
    print("Do not go on to Step 3.")
    print("!" * 72)
    raise SystemExit("STOP: files missing in Step 2")
print()
print("ready: 3 real clips, 7 swap files, 3 matched files, "
      "6 built versions (3 muted, 3 blackout)")

## Step 3 · Write the runner, then check: proof, ladder, sanity

The model never runs inside this notebook's cells directly. It
runs in a plain script, `/content/qwen_equal_run.py`, under this session's own Python (Step 1 installed the stack into the session, so the session interpreter is the environment).
The first cell below writes that script. The second runs it in
**check mode**.

Check mode does three things, in order:

1. **Proof again.** The versions and files the runner imported,
   with the stack pins checked a second time, exactly as the
   Series Two runner proved them.
2. **The loading ladder.** the Series One candidate list under the session Python: `Qwen/Qwen2.5-Omni-3B` in float16, then `Qwen/Qwen2.5-Omni-7B-AWQ`, the talker turned off on each, and the processor from the candidate's repo. Step 1 installs the stack into the session with transformers pinned at 4.52.4 and proves the pin. A failed rung prints its
   full traceback, never a cut-short reason, and appends the same
   words to `equal_qwen_load_log.txt` in the clips folder, so a verbatim record
   survives the session. Whichever rung loads is printed and
   saved with every answer.
3. **The sanity question.** The Series Two gate in its own form:
   one plain text question through the real ask path, and the
   word ready is required in the answer. If the answer comes
   back empty, without the word, or with an error, the script
   stops and Step 4 must not be run. Sound arrival is not this
   gate's job; Step 4's per-ask feed proof carries that.

The ladder, the sanity gate, and the ask path are this model's
Series Two ones, unchanged. Only the battery in Step 4 is new.

In [ ]:
# --- Write the equal battery runner script ----------------------------------------
# The runner holds the proof, the loading candidates, the
# sanity gate, the strict feed proof, and the diagnostic
# battery. It is written to a plain file here and only ever
# run under this session's own Python, the interpreter
# Step 1 installed the stack into.
RUNNER_SOURCE = '\n# qwen_equal_run.py - the equal battery runner for\n# Qwen2.5-Omni-3B. Written by project_k_qwen_equal_colab.ipynb.\n# It runs under the notebook session\'s own Python: the Qwen\n# notebooks install their stack into the Colab session (Step 1),\n# and there is no private environment to run under. Modes:\n# "check" runs the proof, the loading candidates, and the sanity\n# question, then stops. "battery" does all of that and then runs\n# the equal battery tests and saves the CSV.\nimport os\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nimport sys\n\nMODE = sys.argv[1] if len(sys.argv) > 1 else "check"\nprint("runner mode:", MODE)\nprint("runner python:", sys.version.split()[0], sys.executable)\n\n# --- Proof lines, printed before anything loads --------------------------------\n# The versions this script actually imported, from the paths it\n# actually loaded them, repeated here so every run carries its\n# own evidence. The transformers pin is the Series One Qwen pin;\n# anything else stops the run before a model is touched.\nimport torch, transformers\nprint("torch version:", torch.__version__)\nprint("transformers version:", transformers.__version__)\nprint("transformers file:", transformers.__file__)\nimport qwen_omni_utils\nprint("qwen_omni_utils file:", qwen_omni_utils.__file__)\nif transformers.__version__ != "4.52.4":\n    print("STOP: this script is running on transformers",\n          transformers.__version__, "and needs exactly 4.52.4,")\n    print("the version the Series One Qwen notebooks pin.")\n    sys.exit(3)\nprint("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()\n      else "none")\n\n# The yes-score question, settled from the model\'s own code\n# before the run (transformers 4.52.4,\n# models/qwen2_5_omni/modeling_qwen2_5_omni.py):\n# Qwen2_5OmniForConditionalGeneration.generate (line 4412) is a\n# wrapper. It routes every unprefixed keyword through\n# shared_kwargs into the thinker\'s kwargs (lines 4492 and\n# 4496). return_audio defaults to self.has_talker (line 4458),\n# and disable_talker(), which the load below calls exactly as\n# Series One did, sets has_talker to False (lines 4355 to\n# 4360). So generate_audio is False (line 4505) and the wrapper\n# returns self.thinker.generate(...) untouched (lines 4510 to\n# 4513). The thinker is a standard GenerationMixin model, so\n# return_dict_in_generate and output_scores asked of that same\n# greedy call come back on its result as .sequences and\n# .scores, and the first score row is the distribution at the\n# first generated position. The flags cannot change the prompt,\n# the feed, or the tokens greedy decoding chooses; they only\n# keep numbers the call already computes. MiniCPM\'s chat door,\n# by contrast, returns decoded text only, which is why its\n# Series Two rows read "not exposed". Here the score is exposed\n# by the door itself, so YES_SCORE_EXPOSED is True, and yes/no\n# asks record the probability of a yes token at the first\n# generated position from that call\'s first score row.\nYES_SCORE_EXPOSED = True\nprint("yes token score exposed by the generate door:", YES_SCORE_EXPOSED,\n      "(the wrapper returns the thinker\'s own generate result when",\n      "the talker is off, and that result carries its scores)")\n\n# The same source holds one more fact this runner respects: the\n# thinker\'s max_new_tokens is pre-seeded from the wrapper\'s own\n# thinker_max_new_tokens parameter (default 1024), and the\n# merge only fills keys that are missing, so an unprefixed\n# max_new_tokens is silently ignored by this wrapper. The\n# Series One notebooks passed the cap unprefixed; their answers\n# all ended at the end token long before either cap, so their\n# record stands. Here the job list\'s caps are requirements, so\n# every generate call below passes its cap as\n# thinker_max_new_tokens, the wrapper\'s documented knob. The\n# prompt, the feed, and the greedy decoding are untouched.\n\n# --- The load log ----------------------------------------------------------------\n# A failed candidate prints its full traceback (never a\n# cut-short reason) and the same words are appended, as events\n# happen, to equal_qwen_load_log.txt, in the Drive clips\n# folder first and /content second, so the file is already on\n# disk even if the session dies right after a candidate does.\nimport traceback, datetime\n\nLOAD_LOG_PATHS = ["/content/drive/MyDrive/project-k-clips/equal_qwen_load_log.txt",\n                  "/content/equal_qwen_load_log.txt"]\n\ndef log_load(text):\n    for p in LOAD_LOG_PATHS:\n        try:\n            with open(p, "a") as fh:\n                fh.write(text)\n        except Exception:\n            pass   # that home is not mounted; the printed copy stands\n\ndef log_paths_saved():\n    return [p for p in LOAD_LOG_PATHS if os.path.exists(p)]\n\nlog_load("=" * 70 + "\\n")\nlog_load("Qwen2.5-Omni-3B Series Two load log - "\n         + datetime.datetime.now().isoformat(timespec="seconds")\n         + " - runner mode: " + MODE + "\\n")\n\n# --- Load the model: the Series One candidates, unchanged ----------------------\n# Small one first: it is the one that fits a T4-class chip, and\n# it is the model the Series One run loaded. The big AWQ build\n# is tried only if the small one fails, exactly as in the\n# Series One notebook. Either way the weights load in 16-bit\n# form (float16) with device_map="auto", the talker is turned\n# off (written answers only, and it is what makes the wrapper\n# return the thinker\'s result directly), and the processor\n# comes from the same repo. Whichever candidate loads, its\n# name is printed and saved with every answer.\nfrom transformers import (Qwen2_5OmniForConditionalGeneration,\n                          Qwen2_5OmniProcessor)\n\ndef free_memory():\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nmodel = None\nprocessor = None\nMODEL_NAME = None   # repo name of the model that loaded\nLOAD_MODE = None    # the fit that worked\n\nCANDIDATES = [("Qwen/Qwen2.5-Omni-3B", "fp16"),\n              ("Qwen/Qwen2.5-Omni-7B-AWQ", "AWQ weights, fp16 load")]\n\ncandidate_errors = []\nfor repo, mode in CANDIDATES:\n    print("=" * 70)\n    print(f"candidate: {repo} ({mode})")\n    try:\n        m = Qwen2_5OmniForConditionalGeneration.from_pretrained(\n            repo, torch_dtype=torch.float16, device_map="auto")\n        m.disable_talker()  # written answers only; saves memory\n        p = Qwen2_5OmniProcessor.from_pretrained(repo)\n        model, processor = m, p\n        MODEL_NAME, LOAD_MODE = repo, mode\n        print("loaded:", repo)\n        break\n    except Exception:\n        tb = traceback.format_exc()\n        candidate_errors.append((repo, mode, tb))\n        print("this candidate failed. Full traceback:")\n        print(tb)\n        log_load("-" * 70 + "\\n")\n        log_load("candidate failed: " + repo + " (" + mode + ")\\n")\n        log_load(tb)\n        model = None\n        processor = None\n        free_memory()\n\nprint()\nif model is None:\n    print("!" * 70)\n    print("BOTH CANDIDATES FAILED. No model loaded.")\n    saved = log_paths_saved()\n    if saved:\n        print("The full traceback of every candidate is saved, word")\n        print("for word, in:")\n        for p in saved:\n            print("   ", p)\n    else:\n        print("The load log could not be written anywhere; the full")\n        print("tracebacks are in the output above, and nowhere else.")\n    print("Do not go on. Show this whole output to Muse.")\n    print("!" * 70)\n    sys.exit(4)\nprint("MODEL FOR THIS RUN:", MODEL_NAME, "in", LOAD_MODE, "form")\nlog_load("loaded: " + MODEL_NAME + " in " + LOAD_MODE + " form\\n")\nprint("environment: Colab session python (Step 1 installs),",\n      "transformers 4.52.4")\nif torch.cuda.is_available():\n    used = torch.cuda.memory_allocated() / 1e9\n    print(f"chip memory already used by the model: {used:.1f} GB")\n\n# The yes token ids, read from the model\'s own tokenizer (the\n# processor\'s): the first token of "Yes" and of " Yes". A yes\n# score is the summed probability of those ids at the first\n# generated position.\nYES_TOKEN_IDS = []\nfor _word in ("Yes", " Yes"):\n    _enc = processor.tokenizer.encode(_word, add_special_tokens=False)\n    if _enc:\n        YES_TOKEN_IDS.append(int(_enc[0]))\nYES_TOKEN_IDS = sorted(set(YES_TOKEN_IDS))\nprint("yes token ids for scoring:", YES_TOKEN_IDS,\n      [processor.tokenizer.decode([i]) for i in YES_TOKEN_IDS])\n\n# --- Sanity check: can the model write at all? ---------------------------------\n# The Series One gate, in its own form: one plain text question\n# through the real processor and generate path (chat template,\n# processor, generate, decode only the new tokens), with the\n# word ready required in the answer. It proves the model can\n# write through this path before any clip runs. It does not\n# prove sound arrives; the per-ask FEED PROOF in the battery is\n# what proves that. An empty answer, a missing ready, or an\n# error all stop the run here, loudly. Both modes run it.\ntry:\n    convo = [{"role": "user",\n              "content": [{"type": "text",\n                           "text": "Reply with the word ready."}]}]\n    stext = processor.apply_chat_template(convo, add_generation_prompt=True,\n                                          tokenize=False)\n    sinputs = processor(text=stext, return_tensors="pt", padding=True)\n    sinputs = sinputs.to(model.device)\n    with torch.no_grad():\n        sout = model.generate(**sinputs, thinker_max_new_tokens=16,\n                              do_sample=False)\n    snew = sout[:, sinputs.input_ids.shape[1]:]\n    sraw = processor.batch_decode(snew, skip_special_tokens=True,\n                                  clean_up_tokenization_spaces=False)[0]\n    print("SANITY RAW OUTPUT:", repr(sraw))\n    if not sraw.strip():\n        print("WARNING: the sanity check came back EMPTY.")\n        print("STOP. Show this output to Muse.")\n        sys.exit(5)\n    if "ready" not in sraw.lower():\n        print("WARNING: the sanity answer does not contain the word ready.")\n        print("The write path ran, but the model did not follow the one")\n        print("instruction it was given. STOP. Show this output to Muse.")\n        sys.exit(5)\n    print("sanity check passed: the model can write through the real")\n    print("processor and generate path, and the word ready is in its")\n    print("answer.")\nexcept SystemExit:\n    raise\nexcept Exception as e:\n    print("WARNING: the sanity check FAILED with an error:")\n    print(str(e))\n    print("STOP. Show this output to Muse.")\n    sys.exit(5)\n\nif MODE == "check":\n    print()\n    print("check mode done: proof, candidates, and sanity all passed.")\n    print("The equal battery was not run. Run Step 4 for the tests.")\n    sys.exit(0)\n\n# --- Feed proof machinery ---------------------------------------------------------\n# The audit rule: the proof is printed from inside the session,\n# per ask, before the answer is taken, and it measures what the\n# model\'s own feed path actually decoded for that ask. The ask\n# below first calls process_mm_info on the very conversation it\n# is about to feed, exactly as the Series One ask path does, and\n# the proof measures audios[0], the decoded waveform array that\n# call returns and the processor then receives: duration,\n# sample count, RMS in dB, nonzero share, and the same numbers\n# per one-second slice of that waveform. One second is the\n# proof\'s measuring grid here (the same grid the MiniCPM Series\n# Two CSV uses); the processor takes the waveform whole and\n# cuts nothing into seconds, so the padding column records only\n# how short the final slice falls of a full second, as\n# bookkeeping. The sample rate is 16000 by the loader\'s\n# construction (qwen-omni-utils decodes every soundtrack and\n# every audio element at SAMPLE_RATE 16000, mono). The frame\n# count comes from the video tensor the same call returns. A\n# copy of each decoded track is saved beside the results and\n# the track is hashed. The hard-stop thresholds are the MiniCPM\n# and OLA Series Two runners\': a track that should carry sound\n# decoding missing, empty, short by more than 0.5 s against the\n# file\'s own duration, or all zeros stops the run; declared\n# silence is exempt by name and verified to be zeros.\nimport csv, hashlib, json, math, re, subprocess\nimport numpy as np\nimport soundfile as sf\nfrom qwen_omni_utils import process_mm_info\n\nCLIPS = "/content/clips"\nTRACK_DIRS = ["/content/drive/MyDrive/project-k-clips/equal_qwen_extracted_tracks",\n              "/content/equal_qwen_extracted_tracks"]\nfor d in TRACK_DIRS:\n    try:\n        os.makedirs(d, exist_ok=True)\n    except Exception:\n        pass\n\nclass FeedProofError(RuntimeError):\n    pass\n\ndef rms_db(x):\n    if len(x) == 0:\n        return -120.0\n    v = float(np.sqrt(np.mean(np.square(x, dtype=np.float64))))\n    return 20.0 * math.log10(v) if v > 0 else -120.0\n\ndef media_duration(path):\n    out = subprocess.run(\n        ["ffprobe", "-v", "error", "-show_entries", "format=duration",\n         "-of", "csv=p=0", path], capture_output=True, text=True)\n    try:\n        return float(out.stdout.strip())\n    except Exception:\n        return None\n\ndef file_sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as fh:\n        for block in iter(lambda: fh.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n# The feeds, stated exactly. Both ride the model\'s own processor\n# path; the difference between them is the one the notebook\'s\n# title cell states, and it is written into every row.\nFEED_WITH_PICTURE = (\n    "the Series One ask path, unchanged: the conversation "\n    "carries the video file (fps=1, max_pixels=100352, "\n    "min_pixels=3136) and the question; qwen-omni-utils "\n    "process_mm_info with use_audio_in_video=True checks the "\n    "file carries an audio stream (PyAV) and decodes its "\n    "soundtrack with librosa at 16 kHz mono, and samples the "\n    "frames in-process (decord, torchvision fallback); the "\n    "processor builds the text, Whisper feature extractor "\n    "audio features, and video inputs; model.generate is "\n    "called with use_audio_in_video=True, greedy "\n    "(do_sample=False), the token cap passed as "\n    "thinker_max_new_tokens (the wrapper\'s own knob; a plain "\n    "max_new_tokens is ignored by this wrapper), and only the "\n    "newly written tokens are decoded; frames fed this ask: "\n    "{frames}; at most {tokens} new tokens; the proof above "\n    "measures audios[0], the exact decoded waveform array "\n    "this same process_mm_info call handed to the processor")\nFEED_AUDIO_ALONE = (\n    "sound alone through the model\'s own documented audio "\n    "input: the conversation carries an audio element naming "\n    "the same file and no video element; qwen-omni-utils "\n    "process_audio_info decodes an audio element and a video "\n    "soundtrack in the same function with the same librosa "\n    "call at 16 kHz mono, so the soundtrack measured above "\n    "is the same file\'s soundtrack through the same decoder; "\n    "the processor is called with text and audio only (no "\n    "images, no videos, no use_audio_in_video), and "\n    "model.generate runs without the in-video flag, greedy "\n    "(do_sample=False), the token cap passed as "\n    "thinker_max_new_tokens; the difference from the "\n    "with-picture arm is exactly this: no picture reaches "\n    "the model, and the prompt carries an audio token in "\n    "place of video tokens; at most {tokens} new tokens; "\n    "the proof above measures audios[0], the exact decoded "\n    "waveform array this same process_mm_info call handed "\n    "to the processor")\n\ndef yes_probability(scores):\n    # The probability of a yes token at the first generated\n    # position, from the first score row of the same generate\n    # call that produced the answer. Softmax over the\n    # vocabulary, summed over the yes token ids read from the\n    # model\'s own tokenizer at load time.\n    if not scores:\n        return None\n    logits = scores[0][0].float()\n    probs = torch.softmax(logits, dim=-1)\n    total = 0.0\n    for i in YES_TOKEN_IDS:\n        total += float(probs[i].item())\n    return total\n\ndef prove_feed(path, audios, frames_fed, arm, expected_sound,\n               expected_silence):\n    # Measure the waveform array the feed path decoded for this\n    # ask. Raises FeedProofError on any failed check; returns\n    # the proof dict on success. The feed text is filled in by\n    # ask(), which knows which arm is being ridden.\n    if audios is None or len(audios) == 0 or audios[0] is None:\n        raise FeedProofError(\n            f"the feed path returned no audio for {path}")\n    audio_np = np.asarray(audios[0], dtype=np.float32)\n    if audio_np.ndim > 1:\n        audio_np = audio_np[:, 0]\n    sr = 16000   # the loader\'s own rate, by construction\n    src_dur = media_duration(path)\n    if src_dur is None:\n        raise FeedProofError(\n            f"could not read media duration for {path}")\n    track_dur = len(audio_np) / sr if sr else 0.0\n    if len(audio_np) == 0:\n        raise FeedProofError(f"decoded track is empty for {path}")\n    if track_dur < src_dur - 0.5:\n        raise FeedProofError(\n            f"decoded track is short for {path}: {track_dur:.3f}s "\n            f"of track for a {src_dur:.3f}s file")\n    overall_rms = round(rms_db(audio_np), 2)\n    nonzero_share = float(np.count_nonzero(audio_np) / len(audio_np))\n    unit_count = int(math.ceil(len(audio_np) / sr))\n    if unit_count <= 0:\n        unit_count = 1\n    per_unit_rms = []\n    per_unit_padding = []\n    zero_chunks = 0\n    for i in range(unit_count):\n        seg = audio_np[i * sr:(i + 1) * sr]\n        per_unit_rms.append(round(rms_db(seg), 2))\n        per_unit_padding.append(int(sr - len(seg)) if len(seg) < sr\n                                else 0)\n        if len(seg) and not np.any(seg):\n            zero_chunks += 1\n    if expected_sound and (nonzero_share == 0.0 or overall_rms <= -119.5\n                           or zero_chunks == unit_count):\n        raise FeedProofError(\n            f"track that should carry sound decoded silent for "\n            f"{path}: rms={overall_rms} dB, "\n            f"nonzero share={nonzero_share}")\n    if expected_silence and nonzero_share > 0.001:\n        raise FeedProofError(\n            f"declared silence track is not silent for {path}: "\n            f"nonzero share={nonzero_share}")\n    track_hash = hashlib.sha256(audio_np.tobytes()).hexdigest()\n    track_name = (re.sub(r"[^A-Za-z0-9]+", "_", os.path.basename(path))\n                  + "__" + track_hash[:12] + ".wav")\n    track_paths = []\n    for d in TRACK_DIRS:\n        try:\n            tp = os.path.join(d, track_name)\n            if not os.path.exists(tp):\n                sf.write(tp, audio_np, sr, subtype="PCM_16")\n            track_paths.append(tp)\n        except Exception:\n            pass\n    proof = {\n        "file": os.path.basename(path),\n        "source_file_sha256": file_sha256(path),\n        "frame_ffmpeg_rc": ("n/a (frames are decoded in-process "\n                            "by qwen-omni-utils via decord or "\n                            "torchvision; this path makes no "\n                            "ffmpeg call)"),\n        "audio_ffmpeg_rc": ("n/a (the soundtrack is decoded "\n                            "in-process by qwen-omni-utils via "\n                            "librosa; this path makes no ffmpeg "\n                            "call; ffprobe is used only to read "\n                            "the file\'s own duration for the "\n                            "short-track check)"),\n        "media_duration_s": round(src_dur, 3),\n        "track_duration_s": round(track_dur, 3),\n        "sample_rate": int(sr),\n        "sample_count": int(len(audio_np)),\n        "track_rms_db": overall_rms,\n        "track_nonzero_share": round(nonzero_share, 6),\n        "unit_count": int(unit_count),\n        "per_unit_rms_db": per_unit_rms,\n        "per_unit_padding_samples": per_unit_padding,\n        "padding_samples_total": int(sum(per_unit_padding)),\n        "zero_chunk_count": int(zero_chunks),\n        "track_sha256": track_hash,\n        "track_copy_paths": track_paths,\n        "expected_sound": bool(expected_sound),\n        "expected_silence": bool(expected_silence),\n        "frames_fed": int(frames_fed),\n        "arm": arm,\n        "unit_note": ("unit columns count 1 s slices of the "\n                      "decoded waveform handed to the "\n                      "processor; per-unit padding is the "\n                      "final slice\'s shortfall against a full "\n                      "second, proof bookkeeping only"),\n    }\n    return proof\n\ndef ask(job, prompt, max_tokens):\n    # One question about one file, through the model\'s own\n    # processor path. The with-picture arm is the Series One\n    # ask, unchanged. The sound-alone arm rides the documented\n    # audio element form. Either way the feed proof prints\n    # before the answer is taken, and the generate call is the\n    # same greedy call, asked to return its scores so yes/no\n    # asks can record the yes probability from its first score\n    # row.\n    path = os.path.join(CLIPS, job["file"])\n    if not os.path.exists(path):\n        raise FeedProofError(f"feed file missing: {path}")\n    want_yes = prompt.rstrip().endswith("Answer yes or no.")\n    if job["arm"] == "audio_alone":\n        conversation = [{\n            "role": "user",\n            "content": [\n                {"type": "audio", "audio": path},\n                {"type": "text", "text": prompt},\n            ],\n        }]\n        text = processor.apply_chat_template(\n            conversation, add_generation_prompt=True, tokenize=False)\n        audios, images, videos = process_mm_info(\n            conversation, use_audio_in_video=False)\n        proof = prove_feed(path, audios, 0, job["arm"],\n                           job["expected_sound"],\n                           job["expected_silence"])\n        proof["feed"] = FEED_AUDIO_ALONE.format(tokens=max_tokens)\n        print("FEED PROOF " + json.dumps(proof, sort_keys=True))\n        inputs = processor(text=text, audio=audios,\n                           return_tensors="pt", padding=True)\n        inputs = inputs.to(model.device).to(model.dtype)\n        gen_extra = {}\n    else:\n        conversation = [{\n            "role": "user",\n            "content": [\n                {"type": "video", "video": path, "fps": 1,\n                 "max_pixels": 100352, "min_pixels": 3136},\n                {"type": "text", "text": prompt},\n            ],\n        }]\n        text = processor.apply_chat_template(\n            conversation, add_generation_prompt=True, tokenize=False)\n        audios, images, videos = process_mm_info(\n            conversation, use_audio_in_video=True)\n        frames_fed = 0\n        if videos is not None and len(videos):\n            try:\n                frames_fed = int(videos[0].shape[0])\n            except Exception:\n                frames_fed = 0\n        proof = prove_feed(path, audios, frames_fed, job["arm"],\n                           job["expected_sound"],\n                           job["expected_silence"])\n        proof["feed"] = FEED_WITH_PICTURE.format(\n            frames=frames_fed, tokens=max_tokens)\n        print("FEED PROOF " + json.dumps(proof, sort_keys=True))\n        inputs = processor(text=text, audio=audios, images=images,\n                           videos=videos, return_tensors="pt",\n                           padding=True, use_audio_in_video=True)\n        inputs = inputs.to(model.device).to(model.dtype)\n        gen_extra = {"use_audio_in_video": True}\n    with torch.no_grad():\n        out = model.generate(**inputs,\n                             thinker_max_new_tokens=max_tokens,\n                             do_sample=False,\n                             return_dict_in_generate=True,\n                             output_scores=True, **gen_extra)\n    if hasattr(out, "sequences"):\n        seqs = out.sequences\n        scores = getattr(out, "scores", None)\n    elif isinstance(out, tuple):\n        seqs = out[0]\n        scores = None\n    else:\n        seqs = out\n        scores = None\n    # Decode ONLY the tokens the model just wrote, never the\n    # whole sequence, exactly as the Series One ask path does.\n    new_tokens = seqs[:, inputs.input_ids.shape[1]:]\n    decoded = processor.batch_decode(\n        new_tokens, skip_special_tokens=True,\n        clean_up_tokenization_spaces=False)[0].strip()\n    yp = yes_probability(scores) if want_yes else None\n    del audios, images, videos, inputs, out, seqs, new_tokens\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return decoded, proof, yp\n\n# --- Equal battery ---------------------------------------------------------\n# The repaired JSON question, injected at build time from\n# pilot2/run_eggs.py, character for character (444 characters).\nPROMPT_JSON = \'You are given a short first-person video clip. Watch and listen to the whole clip.\\nRespond with ONLY a JSON object in exactly this form, no other text: {"seen": "<the main things happening that you can SEE, short>", "heard": "<the main things happening that you can HEAR, short; write the word none if you hear no sound at all>", "sound_start_s": <number: the second at which the main sound event starts, or the word null if there is no sound>}\'\nassert len(PROMPT_JSON) == 444, f"prompt drifted: {len(PROMPT_JSON)} chars"\n\nprint("FEED SETTINGS, with-picture asks:", FEED_WITH_PICTURE.format(\n    frames="N", tokens="M"))\nprint("FEED SETTINGS, sound-alone asks:", FEED_AUDIO_ALONE.format(\n    tokens="M"))\nprint("MODEL:", MODEL_NAME, "in", LOAD_MODE, "form")\n\n# --- Test definitions: the equal battery (Series One, completed) -----------------\n# The plain sounds question, P3, injected at build time from\n# pilot2/swap/run_swap.py, character for character (143 characters).\nPROMPT_P3 = "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'."\nassert len(PROMPT_P3) == 143, f"P3 drifted: {len(PROMPT_P3)} chars"\n\n# The transcription door for the bed test. New for this battery,\n# and the same words in all four equal-battery runners: bed\n# validity is judged per prompt, and this door gives every model\n# the baseline check in its strongest form.\nTRANSCRIPT_PROMPT = "Write down any words spoken in this clip. If no words are spoken, say \'no words\'."\n\nSWAP_FILES = {\n    "silence": "swap_silence.mp4",\n    "speech": "swap_speech.mp4",\n    "beeps": "swap_beeps.mp4",\n    "water": "swap_water.mp4",\n    "vacuum": "swap_vacuum.mp4",\n    "sizzle": "swap_sizzle.mp4",\n    "mixed": "swap_mixed.mp4",\n}\n\nREAL_CLIPS = [("vacuum", "P02_07_5.mp4"), ("jar", "P32_01_54.mp4"),\n              ("eggs", "own-01.mp4")]\n\nMATCHED_FILES = [("matched_speech", "matched_speech.mp4"),\n                 ("matched_speech_beeps", "matched_speech_beeps.mp4"),\n                 ("matched_speech_vacuum", "matched_speech_vacuum.mp4")]\n\ndef base_job(test, sound, file, arm, version, expected_sound):\n    return {"test": test, "sound": sound, "file": file, "arm": arm,\n            "version": version,\n            "expected_sound": bool(expected_sound),\n            "expected_silence": not bool(expected_sound)}\n\nJOBS = []\n\n# Part A: the repaired JSON question on the three real clips and\n# all seven swap files, two repeats each (20 asks).\nfor sound, file in REAL_CLIPS:\n    for rep in (1, 2):\n        job = base_job("partA_series_one", sound, file,\n                       "audio_with_video", "original", True)\n        JOBS.append((job, "open", "open_json", sound, PROMPT_JSON,\n                     rep, 300))\nfor sound in ("silence", "speech", "beeps", "water", "vacuum",\n              "sizzle", "mixed"):\n    for rep in (1, 2):\n        job = base_job("partA_series_one", sound, SWAP_FILES[sound],\n                       "audio_with_video", "original_swap_still",\n                       sound != "silence")\n        JOBS.append((job, "open", "open_json", sound, PROMPT_JSON,\n                     rep, 300))\n\n# Part A2: the same question on the three real clips muted and\n# blacked out, two repeats each (12 asks). The muted job declares\n# silence; the blackout job still carries the sound.\nfor sound, file in REAL_CLIPS:\n    for version, expected in (("muted", False), ("blackout", True)):\n        for rep in (1, 2):\n            job = base_job("partA2_muted_blackout", sound,\n                           f"equal_{version}_{file}",\n                           "audio_with_video", version, expected)\n            JOBS.append((job, "open", "open_json", sound, PROMPT_JSON,\n                         rep, 300))\n\n# Part B: the plain sounds question on the three matched bed\n# files, two repeats each (6 asks). The first file is the\n# baseline, the sentence alone; this prompt\'s bed rows are read\n# only if this prompt\'s baseline returned the sentence.\nfor sound, file in MATCHED_FILES:\n    for rep in (1, 2):\n        job = base_job("partB_matched_bed", sound, file,\n                       "audio_with_video", "original", True)\n        JOBS.append((job, "direct", "plain_sounds", "speech",\n                     PROMPT_P3, rep, 128))\n\n# Part B door: the transcription question on the same three\n# matched files, one repeat each (3 asks). Same reading rule:\n# the door\'s bed rows are read only if the door\'s baseline\n# returned the sentence.\nfor sound, file in MATCHED_FILES:\n    job = base_job("partB_door_transcription", sound, file,\n                   "audio_with_video", "original", True)\n    JOBS.append((job, "direct", "transcription", "speech",\n                 TRANSCRIPT_PROMPT, 1, 128))\n\nprint(f"equal battery jobs: {len(JOBS)} asks "\n      f"(part A: 20, part A2: 12, part B: 6, part B door: 3)")\n\nFIELDS = ["test", "sound", "file", "arm", "version", "question_form",\n          "question_label", "target", "prompt", "repeat",\n          "target_start_s", "expected_sound", "expected_silence",\n          "frame_ffmpeg_rc", "audio_ffmpeg_rc", "track_duration_s",\n          "sample_rate", "sample_count", "track_rms_db",\n          "track_nonzero_share", "unit_count", "per_unit_rms_db",\n          "per_unit_padding_samples", "padding_samples_total",\n          "zero_chunk_count", "track_sha256", "track_copy_paths",\n          "source_file_sha256", "raw_answer", "yes_score",\n          "yes_score_status", "model", "feed"]\n\nYES_STATUS_EXPOSED = ("exposed: the probability of a yes token at the "\n                      "first generated position, read from the "\n                      "output_scores of the same greedy generate call "\n                      "that produced the answer (the model\'s generate "\n                      "wrapper returns the thinker\'s own result when "\n                      "the talker is off, as in this run); the prompt, "\n                      "the feed, and the decoding are unchanged")\nYES_STATUS_NOT_YESNO = ("not a yes/no ask: this door exposes scores, "\n                        "but this question form has no yes token "\n                        "to score")\nYES_STATUS_MISSING = ("yes/no ask, but no score was recorded for "\n                      "this row; the words or the error in "\n                      "raw_answer are the record")\n\nrows = []\nfailed = 0\n\ndef save_rows():\n    out_fields = FIELDS\n    for out_path in ("/content/drive/MyDrive/project-k-clips/equal_qwen_results.csv",\n                     "/content/equal_qwen_results.csv"):\n        try:\n            with open(out_path, "w", newline="") as fh:\n                w = csv.DictWriter(fh, fieldnames=out_fields)\n                w.writeheader()\n                w.writerows(rows)\n            print("saved:", out_path)\n        except Exception as e:\n            print("could not save", out_path, "-", str(e)[:120])\n\nfor job, form, label, target, prompt, rep, max_tokens in JOBS:\n    print("=" * 70)\n    print(f"{job[\'test\']} - sound={job[\'sound\']} arm={job[\'arm\']} "\n          f"version={job[\'version\']} form={form}/{label} "\n          f"target={target} repeat={rep}")\n    want_yes = prompt.rstrip().endswith("Answer yes or no.")\n    raw_text = ""\n    error = ""\n    proof = None\n    yp = None\n    try:\n        raw, proof, yp = ask(job, prompt, max_tokens)\n        if not raw.strip():\n            error = "ERROR: empty answer returned"\n            failed += 1\n            print("WARNING: this ask came back EMPTY. Recorded as an error,")\n            print("never as silence.")\n            raw_text = error\n        else:\n            raw_text = raw\n            print("RAW ANSWER:")\n            print(raw)\n    except FeedProofError as e:\n        print()\n        print("!" * 70)\n        print("FEED PROOF HARD STOP. The feed for this ask failed its proof:")\n        print(str(e))\n        print("No answer was taken for this ask, and the run stops here.")\n        print("Rows completed before this ask are saved now. Show this")\n        print("whole output to Muse.")\n        print("!" * 70)\n        save_rows()\n        sys.exit(7)\n    except Exception as e:\n        error = "ERROR: " + traceback.format_exc()\n        failed += 1\n        print(f"WARNING: this ask FAILED with an error. The full error is")\n        print("saved in the CSV column raw_answer.")\n        raw_text = error\n        proof = proof or {}\n    p = proof or {}\n    if want_yes:\n        if yp is not None:\n            ys, yss = f"{yp:.4f}", YES_STATUS_EXPOSED\n        else:\n            ys, yss = "", YES_STATUS_MISSING\n    else:\n        ys, yss = "n/a", YES_STATUS_NOT_YESNO\n    rows.append({\n        "test": job["test"], "sound": job["sound"], "file": job["file"],\n        "arm": job["arm"], "version": job["version"],\n        "question_form": form, "question_label": label, "target": target,\n        "prompt": prompt, "repeat": rep,\n        "target_start_s": job.get("target_start_s", ""),\n        "expected_sound": job["expected_sound"],\n        "expected_silence": job["expected_silence"],\n        "frame_ffmpeg_rc": p.get("frame_ffmpeg_rc", ""),\n        "audio_ffmpeg_rc": p.get("audio_ffmpeg_rc", ""),\n        "track_duration_s": p.get("track_duration_s", ""),\n        "sample_rate": p.get("sample_rate", ""),\n        "sample_count": p.get("sample_count", ""),\n        "track_rms_db": p.get("track_rms_db", ""),\n        "track_nonzero_share": p.get("track_nonzero_share", ""),\n        "unit_count": p.get("unit_count", ""),\n        "per_unit_rms_db": json.dumps(p.get("per_unit_rms_db", "")),\n        "per_unit_padding_samples": json.dumps(\n            p.get("per_unit_padding_samples", "")),\n        "padding_samples_total": p.get("padding_samples_total", ""),\n        "zero_chunk_count": p.get("zero_chunk_count", ""),\n        "track_sha256": p.get("track_sha256", ""),\n        "track_copy_paths": json.dumps(p.get("track_copy_paths", "")),\n        "source_file_sha256": p.get("source_file_sha256", ""),\n        "raw_answer": raw_text,\n        "yes_score": ys,\n        "yes_score_status": yss,\n        "model": (f"{MODEL_NAME} ({LOAD_MODE}) "\n                  "[session python, transformers 4.52.4]"),\n        "feed": p.get("feed", ""),\n    })\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nif failed:\n    print()\n    print("!" * 70)\n    print(f"RUN PROBLEM: {failed} ask(s) failed or came back empty. Their")\n    print("raw words or full errors are in the CSV column raw_answer.")\n    print("This run is NOT clean evidence until those asks are explained.")\n    print("!" * 70)\nelse:\n    print()\n    print("Clean run: every ask answered, every feed proved in print first.")\n\nsave_rows()\nif failed:\n    sys.exit(6)\n'
with open("/content/qwen_equal_run.py", "w") as fh:
    fh.write(RUNNER_SOURCE)
print("runner written:", "/content/qwen_equal_run.py", len(RUNNER_SOURCE), "chars")

In [ ]:
# --- Check mode: proof, candidates, sanity ---------------------------------------
# Runs the runner under this session's own Python and streams
# its output here live. A nonzero exit code means the proof, a
# loading candidate, or the sanity gate failed, and the run
# stops in this cell. The first check run downloads the model
# files once; that takes a while and is normal.
import subprocess, sys
proc = subprocess.run([sys.executable, "/content/qwen_equal_run.py", "check"])
print()
print("check exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE CHECK RUN FAILED. The runner's own output above says")
    print("where: a proof line, a loading candidate, or the sanity")
    print("question. Do not run Step 4. Show this output to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: runner check failed")
print("check passed: proof lines right, model loaded, sanity answered.")
print("Go on to Step 4.")

## Step 4 · Run the equal battery

The next cell runs the runner in **battery mode**: proof and
ladder again, sanity again, then the 41 asks of the equal
battery, the completed Series One battery, identical in all
four equal-battery notebooks. Every ask prints its FEED PROOF
line before the model's answer is taken. Watch for it: that
line is the discipline that makes the answers evidence.

**How this model is asked.** the conversation carries the video file at one frame per second with frames capped at 100352 pixels, `process_mm_info` carries the video's own sound track in, and the processor builds the inputs. Token caps travel by the generate wrapper's own thinker setting, as Step 3 of the Series Two notebook explains. Decoding is greedy
everywhere. Token caps per ask are the job list's own: 300 for
the JSON question, 128 for the plain sounds and transcription
questions.

**Part A (20 asks).** The repaired JSON question, word for word
the Series One question, on the three real clips (vacuum, jar,
eggs) and all seven swap files, two repeats each.

**Part A2 (12 asks).** The same question on the three real
clips in the two versions Step 2 built: muted (the picture with
no sound, so any sound the answer reports is invented or
carried by the picture) and blackout (the sound with a black
picture, so the picture cannot help or interfere), two repeats
each.

**Part B (6 asks).** The plain sounds question (the swap
study's P3: "What sounds can you hear in this clip? Describe
only what is audible, without guessing from the pictures. If
the audio is silent, say 'silent'.") on the three matched bed
files, two repeats each.

**Part B door (3 asks).** "Write down any words spoken in this
clip. If no words are spoken, say 'no words'." on the same
three matched files, one repeat each.

**How the bed is read.** Per prompt, never pooled. Each
prompt's bed rows are read only if that same prompt's baseline
(the sentence alone, `matched_speech.mp4`) returned the
sentence. If the plain sounds baseline fails and the
transcription baseline returns the sentence, the bed is read
through the door, and the reverse likewise. Every row stays in
the CSV either way; the rule is applied at write-up time, not
by deleting rows. This replaces the Series One bed design,
where two of three bed tests came out void for want of a
passing baseline: here every model gets the baseline check in
both forms, including its strongest one.

**Hard stops and warnings.** A feed that fails its proof stops
the whole run on the spot (exit code 7), with the rows
completed so far saved first. An ask that errors or comes back
empty is saved with its full error text in `raw_answer` and a
WARNING on screen, never as silence, and the run ends with the
RUN PROBLEM banner (exit code 6). Results save to `equal_qwen_results.csv` in
the Drive clips folder, one row per ask, with the full
feed-proof numbers on every row, in the same 33-column schema
as the Series Two results. The yes-score columns read n/a on
every row: none of these questions is a yes/no ask.

In [ ]:
# --- Battery mode: the equal battery tests --------------------------------
# Same runner, same session Python, streamed live. The runner
# saves the CSV itself, to the Drive clips folder and to
# /content. A nonzero exit code means the pre-flight gates
# failed up front, a feed proof hard-stopped the run, or the
# run finished with broken asks (the banners above say which).
import subprocess, sys
proc = subprocess.run([sys.executable, "/content/qwen_equal_run.py", "battery"])
print()
print("battery exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE BATTERY RUN DID NOT FINISH CLEAN. Read the runner's")
    print("output above: a gate failed, a feed proof hard-stopped the run,")
    print("or some asks are broken and marked in the CSV. Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: battery run not clean")
print("battery done: every ask answered, every feed proved, CSV saved.")

## Closing · How to read the answers

Read this run against this model's own Series One and Series
Two answers, and against the other three models' equal-battery
runs. The battery is identical across the four notebooks: same
files, same questions byte for byte, same repeats. A difference
is a model difference or a feed difference, and the feed
differences are written down, not hidden. This model's feed:
the conversation carries the video file at one frame per second with frames capped at 100352 pixels, `process_mm_info` carries the video's own sound track in, and the processor builds the inputs. Token caps travel by the generate wrapper's own thinker setting, as Step 3 of the Series Two notebook explains.

- **Part A against Part A2.** Clean minus muted is what the
  sound adds to the answer. Blackout is what the sound does
  when the picture cannot carry the answer: a model that names
  the sound on the clean clip but not on the blackout was
  reading the picture, not hearing the sound.
- **Part B under the baseline rule.** Read each prompt's bed
  rows only where that prompt's baseline returned the sentence
  (Step 4 states the rule). A bed that bends the transcription
  while the report denies the bed is the bed test's signature
  finding, and it only counts where the baseline passed.
- **The fit.** The `model` column of every row names the rung
  that loaded. A squeezed fit's answers are that fit's answers.
- **Scope.** Every claim stays capped at this model, this
  feed, these files, this date.

When you finish, the file `equal_qwen_results.csv` in your Drive is the
record. Send it back, and the four equal-battery runs get
written up side by side.